# E-006 — Criterios clásicos de HVI en el ECG (EchoNext)

**Para qué sirve:** mide qué tan bien detectan la HVI las dos reglas que los cardiólogos usan a mano desde hace décadas, **Sokolow-Lyon** y **Cornell**, calculadas sobre el trazado. Es la referencia "trazado + regla simple": si la red neuronal no las supera, no aporta nada sobre la práctica clínica actual. No se entrena nada: las reglas son fijas. Registro en Obsidian: *Experimentos — Modelo IA* → E-006; *Decisiones* → D-006 y D-010.

**Datos:** `MyDrive/modelo-hvi/datos/echonext/` (señales y metadata de test). Licencia restringida de PhysioNet: no compartir la carpeta ni copiarla al vault.

**Salida:** solo agregados → `MyDrive/modelo-hvi/resultados/E-006_criterios_clasicos.json`.

**Reglas:**
- Nunca imprimir ni graficar el ECG de un paciente individual: los gráficos son promedios de muchos pacientes.
- Antes de compartir: *Editar → Borrar todos los resultados*.
- El código se versiona en `github.com/luchoaquinopla/modelo-hvi` (`notebooks/`).

**Cómo correrlo:** *Entorno de ejecución → Ejecutar todas*. Alcanza con CPU.

In [ ]:
# Celda 1 — Carga de test y paso a milivoltios (datos restringidos: solo Drive privado, nunca el vault)
import json
import urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')
CARPETA = Path('/content/drive/MyDrive/modelo-hvi')
DATOS = CARPETA / 'datos' / 'echonext'
SEMILLA = 20260912
ETIQUETA = 'lvwt_gte_13_flag'
TRAMOS_EDAD = [18, 40, 50, 60, 70, 80, 91]

metadata = pd.read_csv(DATOS / 'echonext_metadata_100k.csv')
prueba = metadata[metadata.split == 'test'].reset_index(drop=True)
prueba['tramo_edad'] = pd.cut(prueba.age_at_ecg, TRAMOS_EDAD, right=False).astype(str)
senales = np.load(DATOS / 'EchoNext_test_waveforms.npy')  # (N, 1, 2500, 12): 10 s a 250 Hz, normalizadas por derivación
tabulares = np.load(DATOS / 'EchoNext_test_tabular_features.npy')

# Normalización publicada por EchoNext (repo IntroECG): unidades GE = valor × desvío + media (E-004)
URL_PARAMS = 'https://raw.githubusercontent.com/PierreElias/IntroECG/master/7-EchoNext%20Minimodel/models/echonext_multilabel_minimodel/waveform_normalization_params.json'
params = {k: np.array(v) for k, v in json.loads(urllib.request.urlopen(URL_PARAMS).read()).items()}
UV_POR_UNIDAD = 4.88  # resolución de GE MUSE, verificada en E-004
crudo = senales[:, 0] * params['std'] + params['mean']  # (N, 2500, 12) en unidades GE
mv = (crudo * UV_POR_UNIDAD / 1000).astype(np.float32)
del senales

# Control: las features tabulares de EchoNext siguen el mismo orden de filas que la metadata
alineacion = {col: round(float(max(abs(np.corrcoef(tabulares[:, j], prueba[col])[0, 1]) for j in range(tabulares.shape[1]))), 4)
              for col in ['age_at_ecg', 'ventricular_rate', 'qrs_duration']}
print(len(prueba), mv.shape, 'correlación máxima con la metadata:', alineacion)

In [ ]:
# Celda 2 — Medición de R y S latido a latido (D-010): detectar latidos, medir en cada uno y tomar la mediana
from scipy.signal import find_peaks

HZ = 250
DERIVACIONES = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']
D = {nombre: i for i, nombre in enumerate(DERIVACIONES)}
DESPLAZAMIENTOS = np.arange(-30, 31)  # recorte de cada latido: −120 a +120 ms alrededor del pico
LINEA_BASE = slice(0, 15)             # −120 a −60 ms: antes del QRS, referencia del cero
VENTANA_QRS = slice(15, 56)           # −60 a +100 ms: donde están R y S


def detectar_latidos(x):
    # Latidos = picos de la "energía" (cuánto cambia la señal sumando las 12 derivaciones), separados ≥ 0,3 s
    energia = np.convolve(np.abs(np.diff(x, axis=0)).sum(axis=1), np.ones(10) / 10, mode='same')
    picos, _ = find_peaks(energia, distance=int(0.3 * HZ), height=0.3 * np.percentile(energia, 99))
    fc = 60 * HZ / np.median(np.diff(picos)) if len(picos) > 1 else np.nan
    picos = picos[(picos + DESPLAZAMIENTOS[0] >= 0) & (picos + DESPLAZAMIENTOS[-1] < len(x))]
    latidos = x[picos[:, None] + DESPLAZAMIENTOS]                                   # (latidos, 61, 12)
    # Una derivación "tiene señal" en un latido si su trazo cambia; en ~1 % de EchoNext hay tramos vacíos (esquema parcial)
    con_senal = (np.abs(np.diff(latidos, axis=1)) > 1e-6).mean(axis=1) > 0.2       # (latidos, 12)
    return latidos, con_senal, fc


def medir_ecg(x, base=LINEA_BASE, qrs=VENTANA_QRS, resumen=np.nanmedian):
    latidos, con_senal, fc = detectar_latidos(x)
    if len(latidos) < 3 or not con_senal.any(axis=0).all():
        return None  # sin latidos suficientes, o alguna derivación sin ningún latido con señal
    if base is not None:
        latidos = latidos - np.median(latidos[:, base], axis=1, keepdims=True)
    r = np.where(con_senal, np.clip(latidos[:, qrs].max(axis=1), 0, None), np.nan)  # altura de R por latido y derivación (mV)
    s = np.where(con_senal, np.clip(-latidos[:, qrs].min(axis=1), 0, None), np.nan) # profundidad de S
    mediano = np.stack([np.median(latidos[con_senal[:, j], :, j], axis=0) for j in range(12)], axis=1)
    return resumen(r, axis=0), resumen(s, axis=0), fc, mediano


R = np.full((len(mv), 12), np.nan)
S = np.full((len(mv), 12), np.nan)
FC = np.full(len(mv), np.nan)
latido_promedio = {0: np.zeros((61, 12)), 1: np.zeros((61, 12))}  # suma de latidos medianos por grupo, solo para el gráfico
cantidad = {0: 0, 1: 0}
for i, x in enumerate(mv):
    medicion = medir_ecg(x)
    if medicion is None:
        continue
    R[i], S[i], FC[i], mediano = medicion
    grupo = int(prueba.loc[i, ETIQUETA])
    latido_promedio[grupo] += mediano
    cantidad[grupo] += 1
latido_promedio = {g: latido_promedio[g] / cantidad[g] for g in latido_promedio}
validos = ~np.isnan(R).any(axis=1)

# Control del detector: la frecuencia estimada con los latidos tiene que coincidir con la que midió el equipo
dif_fc = np.abs(FC - prueba.ventricular_rate.to_numpy())
control_detector = {
    'ecg_medidos': int(validos.sum()), 'ecg_sin_medir': int((~validos).sum()),
    'correlacion_fc_con_equipo': round(float(np.corrcoef(FC[validos], prueba.ventricular_rate[validos])[0, 1]), 4),
    'fc_dentro_de_5_lpm_pct': round(float((dif_fc[validos] <= 5).mean() * 100), 1),
}
amplitudes_mediana = {f'{onda}_{d}': round(float(np.nanmedian(M[:, D[d]])), 2)
                      for onda, M, derivs in [('R', R, ['aVL', 'V5', 'V6']), ('S', S, ['V1', 'V3'])] for d in derivs}
print(control_detector)
print('mediana de amplitudes (mV):', amplitudes_mediana)

In [ ]:
# Celda 3 — Reglas clásicas, recorte de EchoNext y comparación con la línea base de E-005
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

assert validos.all(), 'hay ECG sin medir: filtrarlos antes de aplicar las reglas'
y = prueba[ETIQUETA].to_numpy()
hombre = (prueba.sex == 'male').to_numpy()
sokolow = S[:, D['V1']] + np.maximum(R[:, D['V5']], R[:, D['V6']])
cornell = R[:, D['aVL']] + S[:, D['V3']]
UMBRAL_SOKOLOW = 3.5
umbral_cornell = np.where(hombre, 2.8, 2.0)
positivo = {'sokolow_lyon': sokolow >= UMBRAL_SOKOLOW, 'cornell': cornell > umbral_cornell}
positivo['cualquiera'] = positivo['sokolow_lyon'] | positivo['cornell']
# Puntajes para el AUC: Sokolow tal cual; Cornell como margen sobre su corte por sexo
puntajes = {'sokolow_lyon': sokolow, 'cornell': cornell - umbral_cornell}

# Recorte de EchoNext: una derivación "tocó el techo" si alguna muestra quedó justo en el límite publicado
LIMITES = {'V1': 'lowerbound', 'V5': 'upperbound', 'V6': 'upperbound', 'aVL': 'upperbound', 'V3': 'lowerbound'}
tocado = {d: np.isclose(crudo[:, :, D[d]], params[lado][D[d]]).any(axis=1) for d, lado in LIMITES.items()}
recorte = {
    'limite_mv': {d: round(float(params[lado][D[d]] * UV_POR_UNIDAD / 1000), 2) for d, lado in LIMITES.items()},
    'ecg_recortados_pct': {d: {'con_hvi': round(float(t[y == 1].mean() * 100), 1), 'sin_hvi': round(float(t[y == 0].mean() * 100), 1)}
                           for d, t in tocado.items()},
    'sokolow_maximo_posible_mv': round(float((-params['lowerbound'][D['V1']] + max(params['upperbound'][D['V5']], params['upperbound'][D['V6']])) * UV_POR_UNIDAD / 1000), 2),
}

# Referencias de E-005, con la configuración que ya se eligió en su val (acá no se elige nada)
datos = metadata[metadata.split.isin(['train', 'test'])].copy()
datos['sexo_masculino'] = datos.sex.map({'male': 1, 'female': 0})
entrenamiento = datos[datos.split == 'train']
peso_por_paciente = 1 / entrenamiento.groupby('patient_key').patient_key.transform('size')
COLUMNAS = ['age_at_ecg', 'sexo_masculino', 'ventricular_rate', 'pr_interval', 'qrs_duration', 'qt_corrected']
CONTINUAS = [c for c in COLUMNAS if c != 'sexo_masculino']
X_prueba = prueba.assign(sexo_masculino=prueba.sex.map({'male': 1, 'female': 0}))[COLUMNAS]
regresion = make_pipeline(
    ColumnTransformer([('continuas', make_pipeline(SimpleImputer(strategy='median', add_indicator=True), StandardScaler()), CONTINUAS)], remainder='passthrough'),
    LogisticRegression(C=1.0, max_iter=2000),
).fit(entrenamiento[COLUMNAS], entrenamiento[ETIQUETA], logisticregression__sample_weight=peso_por_paciente)
boosting = HistGradientBoostingClassifier(learning_rate=0.05, max_leaf_nodes=15, max_iter=100, early_stopping=False, random_state=SEMILLA)
boosting.fit(entrenamiento[COLUMNAS], entrenamiento[ETIQUETA], sample_weight=peso_por_paciente)
puntajes_referencia = {'regresion_tabular': regresion.predict_proba(X_prueba)[:, 1], 'boosting_tabular': boosting.predict_proba(X_prueba)[:, 1]}


def sens_espec(y, marcado):
    return {'sensibilidad': round(float(marcado[y == 1].mean()), 4), 'especificidad': round(float((~marcado)[y == 0].mean()), 4),
            'marcados_pct': round(float(marcado.mean() * 100), 1)}


def bootstrap_auc(y, puntajes, repeticiones=1000):
    # Remuestreo pareado: los mismos pacientes para todos los puntajes (igual que E-005)
    rng = np.random.default_rng(SEMILLA)
    valores = {n: [] for n in puntajes}
    for _ in range(repeticiones):
        i = rng.integers(0, len(y), len(y))
        if 0 < y[i].sum() < len(y):
            for n, p in puntajes.items():
                valores[n].append(roc_auc_score(y[i], p[i]))
    return {n: np.array(v) for n, v in valores.items()}


def intervalo(v):
    return [round(float(np.percentile(v, 2.5)), 4), round(float(np.percentile(v, 97.5)), 4)]


todos = {**puntajes, **puntajes_referencia}
muestras = bootstrap_auc(y, todos)
resultados = {
    'semilla': SEMILLA, 'etiqueta': ETIQUETA, 'n': int(len(y)), 'positivos': int(y.sum()),
    'control_detector': control_detector, 'alineacion_con_metadata': alineacion,
    'amplitudes_mediana_mv': amplitudes_mediana, 'recorte': recorte,
    'criterios': {
        n: {'auc': round(float(roc_auc_score(y, todos[n])), 4), 'auc_ic95': intervalo(muestras[n]),
            **(sens_espec(y, positivo[n]) if n in positivo else {})}
        for n in todos
    },
    'cualquiera_de_los_dos': sens_espec(y, positivo['cualquiera']),
    'diferencias_auc': {
        f'{a}_menos_{b}': {'media': round(float((muestras[a] - muestras[b]).mean()), 4), 'ic95': intervalo(muestras[a] - muestras[b])}
        for a, b in [('sokolow_lyon', 'cornell'), ('regresion_tabular', 'sokolow_lyon'), ('regresion_tabular', 'cornell'),
                     ('boosting_tabular', 'sokolow_lyon'), ('boosting_tabular', 'cornell')]
    },
    'por_sexo': {s: {n: {'auc': round(float(roc_auc_score(y[m], puntajes[n][m])), 4), **sens_espec(y[m], positivo[n][m])} for n in puntajes}
                 for s, m in [('female', ~hombre), ('male', hombre)]},
    'por_edad': {t: {n: round(float(roc_auc_score(y[m], puntajes[n][m])), 4) for n in puntajes}
                 for t in sorted(prueba.tramo_edad.unique()) for m in [(prueba.tramo_edad == t).to_numpy()]},
}
print(json.dumps({k: resultados[k] for k in ['recorte', 'criterios', 'cualquiera_de_los_dos', 'diferencias_auc']}, indent=1, ensure_ascii=False))

In [ ]:
# Celda 4 — Robustez y calidad: ¿el resultado depende de cómo se midió o de ECG incompletos?
VARIANTES = {
    'base': {},                                                                   # −60 a +100 ms, mediana entre latidos
    'sin_linea_base_local': {'base': None},                                      # confía en la corrección de EchoNext
    'ventana_ancha': {'base': slice(0, 10), 'qrs': slice(10, 61)},               # −80 a +120 ms
    'latidos_mas_altos': {'resumen': lambda v, axis: np.nanpercentile(v, 90, axis=axis)},
}
robustez = {}
for nombre, v in VARIANTES.items():
    medidas = [medir_ecg(x, **v) for x in mv]
    Rv, Sv = np.array([m[0] for m in medidas]), np.array([m[1] for m in medidas])
    sok = Sv[:, D['V1']] + np.maximum(Rv[:, D['V5']], Rv[:, D['V6']])
    cor = Rv[:, D['aVL']] + Sv[:, D['V3']]
    robustez[nombre] = {
        'sokolow_lyon': {'auc': round(float(roc_auc_score(y, sok)), 4), **sens_espec(y, sok >= UMBRAL_SOKOLOW), 'mediana_mv': round(float(np.median(sok)), 2)},
        'cornell': {'auc': round(float(roc_auc_score(y, cor - umbral_cornell)), 4), **sens_espec(y, cor > umbral_cornell), 'mediana_mv': round(float(np.median(cor)), 2)},
    }
resultados['robustez_medicion'] = robustez


# Calidad: ECG con esquema parcial (alguna derivación sin señal en algún cuarto de los 10 s)
def actividad_por_cuarto(x):
    cambia = np.abs(np.diff(x, axis=0)) > 1e-6
    return np.stack([cambia[q * 625:(q + 1) * 625 - 1].mean(axis=0) for q in range(4)], axis=1)  # (12, 4)


actividad = np.array([actividad_por_cuarto(x) for x in mv])
parcial = (actividad < 0.05).any(axis=(1, 2))
completo = ~parcial
resultados['calidad_senal'] = {
    'ecg_esquema_parcial': int(parcial.sum()), 'ecg_esquema_parcial_pct': round(float(parcial.mean() * 100), 2),
    'con_hvi': int(y[parcial].sum()),
    'cuartos_con_senal_en_parciales': {d: [round(float(a), 2) for a in actividad[parcial, D[d]].mean(axis=0)] for d in DERIVACIONES},
    'auc_solo_ecg_completos': {n: round(float(roc_auc_score(y[completo], puntajes[n][completo])), 4) for n in puntajes},
}
print(json.dumps(robustez, indent=1, ensure_ascii=False))
print(json.dumps({k: v for k, v in resultados['calidad_senal'].items() if k != 'cuartos_con_senal_en_parciales'}, indent=1, ensure_ascii=False))

In [ ]:
# Celda 5 — Gráficos (promedios de miles de pacientes, nunca un ECG individual) y guardado
import matplotlib.pyplot as plt

ms = DESPLAZAMIENTOS * 1000 / HZ
colores = {1: '#D85A30', 0: '#1D9E75'}
rotulos = {1: f'Con HVI (n={cantidad[1]})', 0: f'Sin HVI (n={cantidad[0]})'}
fig, ejes = plt.subplots(2, 2, figsize=(12, 8))
for eje, derivacion, onda in [(ejes[0, 0], 'V1', 'S'), (ejes[0, 1], 'V5', 'R')]:
    eje.axvspan(-60, 100, color='0.92', label='Ventana donde se mide')
    for g in (0, 1):
        eje.plot(ms, latido_promedio[g][:, D[derivacion]], color=colores[g], lw=2, label=rotulos[g])
    eje.axhline(0, color='0.5', lw=0.8)
    eje.set(title=f'Latido promedio en {derivacion} (se mide la onda {onda})', xlabel='Tiempo desde el centro del latido (ms)', ylabel='mV')
    eje.legend(fontsize=8)
for eje, valores, corte, titulo in [
    (ejes[1, 0], sokolow, UMBRAL_SOKOLOW, 'Sokolow-Lyon: S(V1) + R(V5 o V6)'),
    (ejes[1, 1], cornell - umbral_cornell, 0.0, 'Cornell: margen sobre su corte por sexo'),
]:
    bins = np.linspace(np.percentile(valores, 0.5), np.percentile(valores, 99.5), 60)
    for g in (0, 1):
        eje.hist(valores[y == g], bins=bins, density=True, alpha=0.55, color=colores[g], label=rotulos[g])
    eje.axvline(corte, color='k', ls='--', lw=1.5, label='Corte clásico')
    if titulo.startswith('Sokolow'):
        eje.axvline(recorte['sokolow_maximo_posible_mv'], color='0.4', ls=':', lw=1.5, label='Máximo posible por el recorte (4,0 mV)')
    eje.set(title=titulo, xlabel='mV', ylabel='Proporción de pacientes')
    eje.legend(fontsize=8)
fig.suptitle('E-006 — Criterios clásicos de HVI en test de EchoNext (promedios, sin pacientes individuales)')
fig.tight_layout()
ruta_figura = CARPETA / 'resultados' / 'E-006_criterios_clasicos.png'
fig.savefig(ruta_figura, dpi=130)
plt.show()

ruta_resultados = CARPETA / 'resultados' / 'E-006_criterios_clasicos.json'
ruta_resultados.write_text(json.dumps(resultados, indent=2, ensure_ascii=False))
print('Guardado:', ruta_resultados.name, ruta_figura.name)
print(json.dumps({k: resultados[k] for k in ['por_sexo', 'por_edad']}, indent=1, ensure_ascii=False))